In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages,"```json")
    text =chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
dataset = generate_dataset()

dataset

[{'task': 'Write a Python function that validates an AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens, must start and end with letter or number)'},
 {'task': 'Create a JSON CloudFormation template snippet that defines a basic AWS Lambda function resource with a Python 3.11 runtime and basic execution role'},
 {'task': 'Write a regular expression that matches valid AWS IAM role ARNs in the format arn:aws:iam::account-id:role/role-name'}]

In [5]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [6]:
# The run_prompt function merges the prompt and test case input, then returns the result
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [7]:
# The run_test_case function calls run_prompt, then grades the result
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [8]:
# The run_eval function loads the dataset and calls run_test_case with each case
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [9]:
# Load the dataset and run the evaluation
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a comprehensive solution:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_s3_region(s3_url: str) -> str:\n    \"\"\"\n    Extract AWS region from an S3 bucket URL.\n    \n    Supports multiple S3 URL formats:\n    - s3://my-bucket.s3.us-west-2.amazonaws.com/key\n    - s3://my-bucket.s3-us-west-2.amazonaws.com/key\n    - https://my-bucket.s3.us-west-2.amazonaws.com/key\n    - https://s3.us-west-2.amazonaws.com/my-bucket/key\n    \n    Args:\n        s3_url: S3 URL string\n        \n    Returns:\n        Region code (e.g., 'us-west-2') or None if not found\n        \n    Raises:\n        ValueError: If URL format is invalid\n    \"\"\"\n    if not s3_url:\n        raise ValueError(\"URL cannot be empty\")\n    \n    # Pattern 1: Virtual-hosted-style (bucket in subdomain)\n    # s3://my-bucket.s3.us-west-2.amazonaws.com or s3://my-bucket.s3-us-west-2.amazonaws.com\n    pattern1 = r'\\.s3[.-]([\\w-